In [ ]:
!nvidia-smi

In [ ]:
import os
# NOTE!
# this is for a multi-gpu setup. we basically set which GPU is visible for the program.
# if you only have one GPU (most likely the case ;) ), make sure this is set to "0"!
# or remove this line completely!
# otherwise you may accidentally make your GPU "invisible" to the program!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Building Modules

This notebook showcases different ways to build larger networks using the `nn.Module` class.
In particular, we showcase the advantages of creating custom subclasses rather than relying on `Sequential`, even in cases where the model _is_ actually just a sequence of layers.

In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt
from torch import nn

## Flat Sequential

This is easy and quick, but the structure is not great:
- There is no hierarchy, and it's hard to see at a glance which part of the module is doing what.
- Layers don't have names, just numbers.
- The "layers" don't correspond to our usual understanding of what one layer is (linear transformation + non-linear activation).
- It's difficult to isolate specific parts of the model.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

n_hidden = 1024

model1 = nn.Sequential(nn.Flatten(),
                       nn.Linear(32*32*3, n_hidden),
                       nn.GELU(),
                       nn.Linear(n_hidden, n_hidden),
                       nn.GELU(),
                       nn.Linear(n_hidden, n_hidden),
                       nn.GELU(),
                       nn.Linear(n_hidden, 10)).to(device)
print(model1)

Let's say that for whatever reason, we want to extract the hidden linear layers from the model -- maybe we want to do something with their weights, for example.
This is quite annoying here.
First, we have to figure out that the model starts with a `Flatten` layer, so we have to exclude that and start with index 1.
We also have to know that after the hidden layers there is just one final linear layer that we need to exclude.
Finally, since we have a pattern of alternating linear and activation modules, we have to take every second layer.

All of this is very brittle -- if any of these aspects change, we have to manually fiddle with the numbers, and it's all rather hard to follow.

In [ ]:
start_index = 1
exclude_index = -1
step = 2

for layer in model1[start_index:exclude_index:step]:
    print(layer)

## Nested Sequential

One "layer" is usually taken to be a sequence of operations like `linear -> activation`.
This is easily represented by a `Sequential` module.
Crucially, this is itself a module, and thus can be part of other `Sequential`s.
In fact, they can be nested arbitrarily deep!

This allows us to more clearly structure the model.
In the `print` output, the nesting is signified by indentation.
However, we still just have numbered modules.
In more complex architectures, this will quickly become hard to follow.

In [ ]:
def fc_layer(n_inputs: int,
             n_outputs: int):
    # optionally we could also take the activation as an argument
    return nn.Sequential(nn.Linear(n_inputs, n_outputs),
                         nn.GELU())


model2 = nn.Sequential(nn.Flatten(),
                       fc_layer(32*32*3, n_hidden),
                       fc_layer(n_hidden, n_hidden),
                       fc_layer(n_hidden, n_hidden),
                       nn.Linear(n_hidden, 10)).to(device)
print(model2)

Our "extract hidden linear layers" has gotten marginally easier:
We no longer have to think about skipping the activations.
Instead, each of the "layers" we extract is a sequence `linear -> activation`, so we simply take the first element of that.

In [ ]:
for layer in model2[start_index:exclude_index]:
    linear = layer[0]
    print(linear)

## Custom Modules

It's often best to explicitly subclass `nn.Module`.
This is a bit more work, but it's usually worth it in the long term.
The model summary is now much clearer, as it uses the names of the classes.
However, the numbered modules in the outermost level are still a bit annoying.

In [ ]:
class FCLayer(nn.Module):
    def __init__(self,
                 n_inputs: int,
                 n_outputs: int):
        super().__init__()
        self.linear = nn.Linear(n_inputs, n_outputs)
        self.activation = nn.GELU()

    def forward(self,
                inputs: torch.Tensor) -> torch.Tensor:
        return self.activation(self.linear(inputs))


model3 = nn.Sequential(nn.Flatten(),
                       FCLayer(32*32*3, n_hidden),
                       FCLayer(n_hidden, n_hidden),
                       FCLayer(n_hidden, n_hidden),
                       nn.Linear(n_hidden, 10)).to(device)
print(model3)

Extracting the linear layers is once again slightly clearer, as we can replace the indexing by the `.linear` attribute.

In [ ]:
for layer in model3[start_index:exclude_index]:
    linear = layer.linear
    print(linear)

## Fully Custom

We could completely remove `Sequential`, or at least move it a level down in the hierarchy.
A common way to think of/structure neural networks is to decompose them into a "root" or "stem", a "body", and a "head":
- The root should prepare the input for the body in some way, e.g. by mapping to the correct dimensionality. This is often a small part of the network.
- The body does most of the work for feature extraction, and generally has a rather uniform structure, e.g. repeating the same building blocks over and over.
- The head transforms the body output to the final network output. This is usually also small, e.g. just a single classification layer.

This once again requires more code, but makes our networks more understandable and reusable.

In [ ]:
class MLP(nn.Module):
    def __init__(self,
                 n_inputs: int,
                 n_hidden: int,
                 n_outputs: int,
                 n_layers: int):
        super().__init__()
        self.root = nn.Flatten()
        
        self.body = nn.Sequential()
        for ind in range(n_layers):
            self.body.append(FCLayer(n_inputs if ind == 0 else n_hidden, n_hidden))
            
        self.head = nn.Linear(n_hidden, n_outputs)

    def forward(self,
                inputs: torch.Tensor) -> torch.Tensor:
        return self.head(self.body(self.root(inputs)))


model4 = MLP(n_inputs=32*32*3, n_hidden=n_hidden, n_outputs=10, n_layers=3)
print(model4)

For getting the linear layers, we can finally remove the start and end indices and simply get the `body` directly:

In [ ]:
for layer in model4.body:
    linear = layer.linear
    print(linear)